In [ ]:
# Project paths
project_root = "/datasetfolderpath/anonymous"

# Wild Dataset Base Path
wild_base_path = f"{project_root}/LoTE-Animal/wild"

# Web Dataset Base Path
web_base_path = f"{project_root}/LoTE-Animal/web"

train_image_dir = f"{web_base_path}/train/images"
train_label_dir = f"{web_base_path}/train/labels"
val_image_dir   = f"{web_base_path}/val/images"
val_label_dir   = f"{web_base_path}/val/labels"
test_image_dir  = f"{web_base_path}/test/images"
test_label_dir  = f"{web_base_path}/test/labels"


# YAML Save Paths
wild_yaml_path = f"{project_root}/rtdetr_training/working/wild_dataset.yaml"
web_yaml_path  = f"{project_root}/rtdetr_training/working/web_dataset.yaml"

# Trained Model Path
best_model_path = f"{project_root}/rtdetr_training/working/web/runs/detect/train/weights/best.pt"

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install ultralytics opencv-python-headless matplotlib pandas

In [ ]:
import os
import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import RTDETR, YOLO

In [ ]:
# Verify Dataset Structure
print("\nVerifying dataset structure")

print("Dataset Structure:")
print("-" * 20)

directories_to_check = [
    ('Train Images', train_image_dir),
    ('Train Labels', train_label_dir),
    ('Val Images', val_image_dir),
    ('Val Labels', val_label_dir),
    ('Test Images', test_image_dir),
    ('Test Labels', test_label_dir)
]

for name, dir_path in directories_to_check:
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith(('.jpg', '.png', '.jpeg', '.txt'))]
        file_type = "images" if 'image' in name.lower() else "labels"
        print(f"{name}: {len(files)} {file_type}")
        if files:
            print(f"   Sample: {files[0]}")
    else:
        print(f"{name}: Directory not found - {dir_path}")

# Check label format and classes
print("\nChecking label format and classes...")
if os.path.exists(train_label_dir):
    label_files = [f for f in os.listdir(train_label_dir) if f.endswith('.txt')]
    if label_files:
        sample_label_path = os.path.join(train_label_dir, label_files[0])
        with open(sample_label_path, 'r') as f:
            sample_content = f.readlines()
        if sample_content:
            print(f" Sample label format: {sample_content[0].strip()}")
        
        class_ids = set()
        for label_file in label_files[:20]: 
            with open(os.path.join(train_label_dir, label_file), 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            class_id = int(line.strip().split()[0])
                            class_ids.add(class_id)
                        except:
                            continue
        
        print(f" Classes found in labels: {sorted(class_ids)}")
        num_classes = max(class_ids) + 1 if class_ids else 11
        print(f" Number of classes: {num_classes}")
    else:
        num_classes = 11
        print("  No label files found, using default: 11 classes")
else:
    num_classes = 11
    print("  Labels directory not found, using default: 11 classes")

In [ ]:
# Create Dataset YAML
print("\nCreating dataset YAML configuration!")

class_names = [
    'Pandas','RedPandas','MartesFlavigula','MacacaThibetana',
    'RhinopithecusRoxellana','HystrixBrachyura','SusScrofa',
    'RusaUnicolor','ElaphodusCephalophus','CapricornisMilneedwardsii',
    'PseudoisNayaur'
    ]

dataset_yaml = f"""
path: {web_base_path}
train: train/images
val: val/images
test: test/images

nc: 11
names: {class_names}
"""

with open(web_yaml_path, 'w') as f:
    f.write(dataset_yaml)
print(" Dataset YAML created:", web_yaml_path)

print("\nDataset YAML Content:")
print("-" * 20)
print(dataset_yaml)

In [ ]:
# Training Checkpoint Handling
resume_path = "/datasetfolderpath/anonymous/rtdetr_training/working/web/runs/detect/train/weights/last.pt"

if os.path.exists(resume_path):
    print("\nCheckpoint found! Resuming training from last.pt")
    model = RTDETR(resume_path)
    resume_flag = True
else:
    print("\nNo checkpoint found. Starting fresh training!")
    model = RTDETR("rtdetr-l.pt")
    resume_flag = False

In [ ]:
results = model.train(
    data=web_yaml_path,
    epochs=500,
    imgsz=800,
    batch=16,
    device=0,
    workers=4,
    patience=30,
    save=True,
    pretrained=True,
    optimizer='auto',
    lr0=0.001,
    weight_decay=0.0005,
    exist_ok=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    flipud=0.01,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.1,
    copy_paste=0.1,
    resume=resume_flag    
)

print("\nTraining Complete!")
if resume_flag:
    print("Training continued from previous checkpoint.")
else:
    print("Training started fresh.")

In [ ]:
print("\nModel validation on test set!")
print("-" * 20)

model = RTDETR("/datasetfolderpath/anonymous/rtdetr_training/working/web/runs/detect/train/weights/best.pt")

test_metrics = model.val(
    data=web_yaml_path,
    split='test',
    imgsz=800
)

print("\nValidation Results:")
print(f" mAP50:    {test_metrics.box.map50:.4f}")
print(f" mAP50-95: {test_metrics.box.map:.4f}")
print(f" Precision: {test_metrics.box.p.mean():.4f}")
print(f" Recall:    {test_metrics.box.r.mean():.4f}")

In [ ]:
# Load a model
model = RTDETR("/datasetfolderpath/anonymous/rtdetr_training/working/web/runs/detect/train/weights/best.pt")

# Perform object detection on an image
results = model("/datasetfolderpath/anonymous/LoTE-Animal/web/test/images/000000000047.jpg", save=True)
results[0].show()

In [ ]:
# Run Inference
print("\n Running inference on sample images!")
print("-" * 40)

if os.path.exists(test_image_dir):
    sample_images = [f for f in os.listdir(test_image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))][:3]
    
    if sample_images:
        for img_name in sample_images:
            print(f"\n Processing: {img_name}")
            results = model.predict(
                source=os.path.join(test_image_dir, img_name),
                conf=0.5,
                save=True,
                save_txt=True,
                imgsz=800,
                project='/datasetfolderpath/anonymous/rtdetr_training/working/predictions',
                name='test_samples',
                exist_ok=True,
                verbose=False
            )
            
            if results[0].boxes is not None and len(results[0].boxes) > 0:
                num_detections = len(results[0].boxes)
                print(f"   Detections: {num_detections}")
                
                # Get unique classes detected with counts and max confidence
                detected_classes = {}
                for box in results[0].boxes:
                    class_id = int(box.cls[0])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class_{class_id}"
                    confidence = float(box.conf[0])
                    
                    if class_name not in detected_classes:
                        detected_classes[class_name] = {'count': 0, 'max_conf': 0}
                    
                    detected_classes[class_name]['count'] += 1
                    detected_classes[class_name]['max_conf'] = max(detected_classes[class_name]['max_conf'], confidence)
                
                # Print detection summary
                for class_name, info in detected_classes.items():
                    print(f"   - {class_name}: {info['count']} detections (max conf: {info['max_conf']:.3f})")
            else:
                print("   No detections")
    else:
        print("No test images found")
else:
    print("Test directory not found")


In [ ]:
# Create Dataset YAML
print("\nCreating dataset YAML configuration!")

class_names = [
    'Pandas','RedPandas','MartesFlavigula','MacacaThibetana',
    'RhinopithecusRoxellana','HystrixBrachyura','SusScrofa',
    'RusaUnicolor','ElaphodusCephalophus','CapricornisMilneedwardsii',
    'PseudoisNayaur'
    ]

dataset_yaml = f"""
path: {wild_base_path}
train: train/images
val: val/images
test: test/images

nc: 11
names: {class_names}
"""

with open(wild_yaml_path, 'w') as f:
    f.write(dataset_yaml)
print(" Dataset YAML created:", wild_yaml_path)

print("\nDataset YAML Content:")
print("-" * 20)
print(dataset_yaml)

In [ ]:
print("\nModel validation on test set!")
print("-" * 20)

model = RTDETR("/datasetfolderpath/anonymous/rtdetr_training/working/web/runs/detect/train/weights/best.pt")

test_metrics = model.val(
    data=wild_yaml_path,
    split='test',
    imgsz=800
)

print("\nValidation Results:")
print(f" mAP50:    {test_metrics.box.map50:.4f}")
print(f" mAP50-95: {test_metrics.box.map:.4f}")
print(f" Precision: {test_metrics.box.p.mean():.4f}")
print(f" Recall:    {test_metrics.box.r.mean():.4f}")

In [ ]:

print("\n Training logs and summary")
print("-" * 20)

print(" Final Training Results:")
print(f" Model: RTDETR")
print(f" Best model: /datasetfolderpath/anonymous/rtdetr_training/working/web/runs/detect/train/weights/best.pt")
print(f" Final mAP50: {test_metrics.box.map50:.4f}")
print(f" Final mAP50-95: {test_metrics.box.map:.4f}")

# FIXED precision & recall
precision = float(np.mean(test_metrics.box.p))
recall = float(np.mean(test_metrics.box.r))
print(f" Precision: {precision:.4f}")
print(f" Recall: {recall:.4f}")

class_performance = [
    (class_names[i] if i < len(class_names) else f"Class_{i}",
     test_metrics.box.ap50[i])
    for i in range(num_classes)
]
sorted_performance = sorted(class_performance, key=lambda x: x[1], reverse=True)

print("\n Top 5 Performing Classes:")
for i, (class_name, ap50) in enumerate(sorted_performance[:5]):
    print(f"   {i+1}. {class_name}: {ap50:.4f}")

print("\n Lowest 5 Performing Classes:")
for i, (class_name, ap50) in enumerate(sorted_performance[-5:]):
    print(f"   {i+1}. {class_name}: {ap50:.4f}")


# Save YAML
results_summary = {
    'model': 'RTDETR',
    'dataset': 'LoTE-Animal Web',
    'overall_mAP50': float(test_metrics.box.map50),
    'overall_mAP': float(test_metrics.box.map),
    'precision': precision,
    'recall': recall,
    'best_class': sorted_performance[0][0],
    'best_class_ap50': float(sorted_performance[0][1]),
    'worst_class': sorted_performance[-1][0],
    'worst_class_ap50': float(sorted_performance[-1][1])
}

with open('/datasetfolderpath/anonymous/rtdetr_training/working/web/rtdetr_training_summary.yaml', 'w') as f:
    yaml.dump(results_summary, f)

# FIXED: per-class CSV
per_class_results = pd.DataFrame({
    'Class_ID': range(num_classes),
    'Class_Name': [
        class_names[i] if i < len(class_names) else f"Class_{i}"
        for i in range(num_classes)
    ],
    'AP50': test_metrics.box.ap50[:num_classes],
    'AP': test_metrics.box.ap[:num_classes]
})
per_class_results.to_csv('/datasetfolderpath/anonymous/rtdetr_training/working/rtdetr_per_class_results.csv', index=False)

